# Explore a run in `out/`

Point `RUN` at a folder like `ap1_ae5_rpm3333_f40_comp` and run the notebook top
to bottom. Everything is read from what `main.py` left on disk — nothing is
recomputed, except the receptance in section 10 and the spectrum in section 7,
which are derived from `plant.npz` and the deviation trace.

| in the folder | what it holds |
|---|---|
| `summary.json` / `summary.txt` | every scalar the run produced, and the console report |
| `config.json` | the `RunConfig` the run was built from |
| `plant.npz` | A/B/C/D of the linearised arm at the tool tip (SI: N -> m) |
| `stability/` | one row per path node: growth rate, `ap_crit`, linear force `F0` |
| `forces/` | the feedforward `F0(s)`, and one row per spindle revolution, sim vs linear |
| `raw/` | the full coupled simulation, and a decimated CSV of it |

In [ ]:
"""Setup — paths, palette, plotly theme, helpers."""
import json
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, display
from plotly.subplots import make_subplots

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "out").is_dir())
OUT = ROOT / "out"

# --- palette -----------------------------------------------------------------
# Categorical slots are used in fixed order and never cycled. Growth rate gets a
# diverging blue<->red pinned at zero (stable <-> unstable); plain magnitudes get
# the one-hue blue ramp; status colours are reserved and always carry a word.
_STATUS = dict(good="#0ca30c", warning="#fab219", serious="#ec835a", critical="#d03b3b")
LIGHT = dict(
    surface="#fcfcfb", plane="#f9f9f7", ink="#0b0b0b", ink2="#52514e", muted="#898781",
    grid="#e1e0d9", axis="#c3c2b7", wash="rgba(42,120,214,0.07)", ring="rgba(11,11,11,0.10)",
    series=["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"],
    diverging=[[0.0, "#0d366b"], [0.25, "#2a78d6"], [0.5, "#f0efec"], [0.75, "#e34948"], [1.0, "#7a1d1d"]],
    sequential=[[0.0, "#cde2fb"], [0.5, "#2a78d6"], [1.0, "#0d366b"]],
    **_STATUS,
)
DARK = dict(
    surface="#1a1a19", plane="#0d0d0d", ink="#ffffff", ink2="#c3c2b7", muted="#898781",
    grid="#2c2c2a", axis="#383835", wash="rgba(57,135,229,0.12)", ring="rgba(255,255,255,0.10)",
    series=["#3987e5", "#d95926", "#199e70", "#c98500", "#d55181", "#008300", "#9085e9", "#e66767"],
    diverging=[[0.0, "#104281"], [0.25, "#3987e5"], [0.5, "#383835"], [0.75, "#e66767"], [1.0, "#b03434"]],
    sequential=[[0.0, "#184f95"], [0.5, "#3987e5"], [1.0, "#cde2fb"]],
    **_STATUS,
)
P = LIGHT
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'
W = 2  # line width, everywhere
DASHES = ["solid", "dash", "dot", "dashdot"]  # so runs that coincide stay legible


def set_theme(dark=False):
    """Light or dark. Re-run the plot cells after switching."""
    global P
    P = DARK if dark else LIGHT
    ax = dict(gridcolor=P["grid"], zerolinecolor=P["axis"], zerolinewidth=1,
              linecolor=P["axis"], ticks="outside", ticklen=4, tickcolor=P["axis"],
              title_font=dict(color=P["ink2"], size=12), automargin=True)
    pio.templates["milling"] = go.layout.Template(layout=go.Layout(
        paper_bgcolor=P["surface"], plot_bgcolor=P["surface"], colorway=P["series"],
        font=dict(family=FONT, size=13, color=P["ink2"]),
        title=dict(font=dict(size=16, color=P["ink"]), x=0.0, xanchor="left"),
        margin=dict(l=70, r=30, t=70, b=54),
        hoverlabel=dict(font=dict(family=FONT, size=12), bgcolor=P["surface"],
                        bordercolor=P["axis"]),
        # The legend keeps plotly's own place — outside the plot, top right — so it
        # can never land on a subplot title or on the data.
        legend=dict(yanchor="top", y=1.0, xanchor="left", x=1.01,
                    bgcolor="rgba(0,0,0,0)", font=dict(color=P["ink2"], size=12)),
        xaxis=ax, yaxis=ax,
    ))
    pio.templates.default = "milling"


set_theme(dark=False)


def show(fig, title=None, subtitle=None, height=None, **kw):
    """Title, a muted second line under it, size, then draw."""
    t = {}
    if title:
        t["text"] = title
    if subtitle:
        t["subtitle"] = dict(text=subtitle, font=dict(size=12, color=P["muted"]))
    if t:
        fig.update_layout(title=t, margin_t=100 if subtitle else 70)
    if height:
        fig.update_layout(height=height)
    if kw:
        fig.update_layout(**kw)
    fig.show()


def spans(x, flag):
    """Contiguous [x0, x1] stretches where `flag` is True — for shading."""
    flag = np.asarray(flag, bool)
    x = np.asarray(x, float)
    out, i = [], 0
    while i < len(flag):
        if flag[i]:
            j = i
            while j + 1 < len(flag) and flag[j + 1]:
                j += 1
            if j > i:
                out.append((x[i], x[j]))
            i = j + 1
        else:
            i += 1
    return out


def shade(fig, x, flag, label=None, **rc):
    """A wash over the engaged stretches, so every x-axis reads the same."""
    for k, (x0, x1) in enumerate(spans(x, flag)):
        kw = dict(rc)
        if label and k == 0:
            kw.update(annotation_text=label, annotation_position="top left",
                      annotation_font=dict(size=11, color=P["muted"]))
        fig.add_vrect(x0=x0, x1=x1, fillcolor=P["wash"], line_width=0, layer="below", **kw)
    return fig


def subtitles(fig, n):
    """make_subplots writes its titles as annotations — left-align and mute them."""
    for an in fig.layout.annotations[:n]:
        an.update(font=dict(size=12, color=P["ink2"]), x=0, xanchor="left")


def second_legend(fig, y):
    """A legend of its own for the lower subplot, so its names sit beside its rows.

    Traces opt in with `legend="legend2"`. Without this the two rows share one
    list, and a colour appears twice under two different names.
    """
    fig.update_layout(legend2=dict(yanchor="top", y=y, xanchor="left", x=1.01,
                                   bgcolor="rgba(0,0,0,0)",
                                   font=dict(color=P["ink2"], size=12)))

## Load

`load_run` reads a whole run folder. The 5–6 MB `raw/coupled.npz` stays on disk
until you ask for it with `r.coupled`; everything below uses the CSVs and the
small npz files.

In [ ]:
def _npz(path):
    with np.load(path, allow_pickle=True) as z:
        return {k: z[k] for k in z.files}


@dataclass
class Run:
    name: str
    dir: Path
    summary: dict
    config: dict
    report: str
    plant: dict
    stab: dict               # the whole StabilityAlongPath
    along: pd.DataFrame      # one row per path node
    revs: pd.DataFrame       # one row per spindle revolution
    ts: pd.DataFrame         # the simulated pass, decimated
    ff: dict = None          # feedforward — absent on an uncompensated run
    _coupled: dict = field(default=None, repr=False)

    @property
    def coupled(self):
        """The full SimResult — every sample, every joint. Read on first use."""
        if self._coupled is None:
            self._coupled = _npz(self.dir / "raw" / "coupled.npz")
        return self._coupled

    @property
    def dt(self):
        """Sample step of `ts` — the CSV is decimated by summary['csv_decimate']."""
        return float(np.median(np.diff(self.ts["t_s"].to_numpy())))

    @property
    def rev_window(self):
        """How many rows of `ts` one spindle revolution spans."""
        return max(1, int(round((60.0 / self.summary["spindle_rpm"]) / self.dt)))

    @property
    def cutting(self):
        """Rows of `ts` inside the cut.

        The force falls to zero between teeth, so a bare `F > 0` test chops the
        pass into hundreds of stripes — and, taken as a mask, would resample the
        trace unevenly and ruin a spectrum. Widening it to a revolution closes the
        tooth gaps while keeping any real break in the cut.
        """
        return (self.ts["F_mag_N"].rolling(self.rev_window, center=True, min_periods=1)
                .max().to_numpy() > 0.0)

    @property
    def ap_mm(self):
        return float(self.summary["axial_depth_mm"])


def available(root=OUT):
    return sorted(p.name for p in Path(root).iterdir() if (p / "summary.json").is_file())


def load_run(name, root=OUT):
    d = Path(root) / name
    if not (d / "summary.json").is_file():
        raise FileNotFoundError(f"{d} is not a run folder — have: {available(root)}")
    ffp = d / "forces" / "feedforward.npz"
    return Run(
        name=name, dir=d,
        summary=json.loads((d / "summary.json").read_text(encoding="utf-8")),
        config=json.loads((d / "config.json").read_text(encoding="utf-8")),
        report=(d / "summary.txt").read_text(encoding="utf-8"),
        plant=_npz(d / "plant.npz"),
        stab=_npz(d / "stability" / "stability.npz"),
        along=pd.read_csv(d / "stability" / "along_path.csv"),
        revs=pd.read_csv(d / "forces" / "revolutions.csv"),
        ts=pd.read_csv(d / "raw" / "timeseries.csv"),
        ff=_npz(ffp) if ffp.is_file() else None,
    )


print("runs in", OUT)
for n in available():
    print("   ", n)

In [ ]:
RUN = available()[0]           # <- the folder to look at
r = load_run(RUN)

print(r.name, "|", "compensated" if r.summary["compensated"] else "plain")
print(f"{len(r.along)} path nodes, {int(r.along['engaged'].sum())} engaged"
      f" | {len(r.ts)} rows of time series at {r.dt * 1e3:.2f} ms"
      f" (decimated {r.summary['csv_decimate']}x)"
      f" | {len(r.revs)} revolutions")

## 1 · The headline

The scalars the run is judged on. `growth rate` is the exponential rate of the
chatter mode — negative is stable. `ap_crit` is the depth at which the trimmed
cut would tip over; compare it with the depth actually run.

In [ ]:
def headline(r):
    s = r.summary

    def tile(label, value, sub="", color=None):
        return (f'<div style="background:{P["surface"]};border:1px solid {P["ring"]};'
                f'border-radius:10px;padding:14px 16px;min-width:0">'
                f'<div style="font-size:11px;letter-spacing:.06em;text-transform:uppercase;'
                f'color:{P["muted"]}">{label}</div>'
                f'<div style="font-size:23px;line-height:1.25;color:{color or P["ink"]};'
                f'margin-top:4px">{value}</div>'
                f'<div style="font-size:12px;color:{P["ink2"]};margin-top:3px">{sub}</div></div>')

    def verdict(unstable):
        return ("▲ unstable" if unstable else "● stable",
                P["critical"] if unstable else P["good"])

    pv, pc = verdict(s["pred_unstable_trim"])
    sv, sc = verdict(s["sim_unstable"])
    agree = s["agree_trim"]
    tiles = [
        tile("prediction", pv,
             f'growth {s["pred_growth_max_trim_1_s"]:+.2f} 1/s at {s["pred_mode_hz"]:.1f} Hz', pc),
        tile("simulation", sv,
             f'growth {s["sim_growth_1_s"]:+.2f} 1/s at {s["sim_chatter_hz"]:.1f} Hz'
             f' (R² {s["sim_growth_r2"]:.2f})', sc),
        tile("they " + ("agree" if agree else "disagree"),
             "● yes" if agree else "▲ no",
             f'growth differs by {s["growth_err_trim_1_s"]:+.2f} 1/s',
             P["good"] if agree else P["critical"]),
        tile("ap_crit", f'{s["pred_ap_crit_trim_mm"]:.1f} mm',
             f'running at {r.ap_mm:g} mm · min over nodes {s["pred_ap_crit_min_mm"]:.2f} mm'),
        tile("deviation", f'{s["sim_dc_dev_um"]:.1f} µm DC',
             f'AC {s["sim_ac_rms_um"]:.1f} µm rms · peak {s["sim_peak_dev_um"]:.1f} µm'),
        tile("force", f'{s["plateau_F_sim_N"]:.0f} N plateau',
             f'peak |F| {s["sim_peak_force_N"]:.0f} N · '
             f'engine vs linear {s["plateau_mag_err"] * 100:+.2f} %'),
        tile("feedforward",
             f'{s["ff_axes"]}, gain {s["ff_gain"]:g}' if s["ff_applied"] else "off",
             (f'F0 mean {s["ff_F0_mean_N"]:.0f} N · would pull '
              f'{s["ff_offset_max_um"]:.0f} µm' if s["ff_applied"] else "no compensation")),
        tile("cut", f'ap {s["axial_depth_mm"]:g} · ae '
                    f'{r.config["mill"]["radial_engagement_mm"]:g} mm',
             f'{s["spindle_rpm"]:.0f} rpm · fz {s["fz_mm"]:.3f} mm/tooth · '
             f'{s["sim_engaged_s"]:.2f} s engaged'),
    ]
    display(HTML(
        f'<div style="font-family:{FONT};color:{P["ink"]};background:{P["plane"]};'
        f'padding:16px;border-radius:12px">'
        f'<div style="font-size:15px;font-weight:600;margin-bottom:12px">{r.name}</div>'
        f'<div style="display:grid;gap:10px;'
        f'grid-template-columns:repeat(auto-fit,minmax(215px,1fr))">'
        + "".join(tiles) + '</div></div>'))


headline(r)

In [ ]:
# The console report, kept verbatim — the same numbers, in prose.
print(r.report)

## 2 · The cut, on the part

Every path node in the workpiece frame. Colour is the predicted growth rate on a
diverging scale pinned at zero: blue stable, red not. Hollow markers are nodes
the tool never engages. `by="ap_crit"` or `by="F0"` re-colours the same map on
the one-hue magnitude ramp.

In [ ]:
def fig_map(r, by="growth"):
    a = r.along
    eng = a["engaged"].to_numpy(bool)
    col, cbar, scale = {
        "growth": ("growth_rate_1_s", "growth rate (1/s)", "diverging"),
        "ap_crit": ("ap_crit_mm", "ap_crit (mm)", "sequential"),
        "F0": ("F0_mag_N", "|F0| (N)", "sequential"),
    }[by]
    v = a.loc[eng, col].to_numpy(float)
    kw = dict(colorscale=P[scale])
    if scale == "diverging":
        m = float(np.nanmax(np.abs(v))) if len(v) else 1.0
        kw.update(cmin=-m, cmax=m, cmid=0.0)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=r.ts["cmd_x_mm"], y=r.ts["cmd_y_mm"], mode="lines", name="commanded path",
        line=dict(color=P["axis"], width=1), hoverinfo="skip", showlegend=False))
    fig.add_trace(go.Scatter(
        x=a.loc[~eng, "x_mm"], y=a.loc[~eng, "y_mm"], mode="markers", name="not engaged",
        marker=dict(size=8, color="rgba(0,0,0,0)", line=dict(color=P["axis"], width=1.5)),
        customdata=a.loc[~eng, "s_mm"], showlegend=False,
        hovertemplate="s %{customdata:.0f} mm · not engaged<extra></extra>"))
    fig.add_trace(go.Scatter(
        x=a.loc[eng, "x_mm"], y=a.loc[eng, "y_mm"], mode="markers", name="engaged",
        marker=dict(size=11, color=v, line=dict(color=P["surface"], width=1),
                    colorbar=dict(title=dict(text=cbar, side="right"), thickness=12,
                                  outlinewidth=0, len=0.9), **kw),
        customdata=np.c_[a.loc[eng, "s_mm"], a.loc[eng, "ae_mm"], a.loc[eng, "growth_rate_1_s"],
                         a.loc[eng, "ap_crit_mm"], a.loc[eng, "F0_mag_N"], a.loc[eng, "mode_hz"]],
        showlegend=False,
        hovertemplate=("s %{customdata[0]:.0f} mm · ae %{customdata[1]:.2f} mm<br>"
                       "growth %{customdata[2]:+.2f} 1/s at %{customdata[5]:.1f} Hz<br>"
                       "ap_crit %{customdata[3]:.2f} mm · |F0| %{customdata[4]:.0f} N"
                       "<extra></extra>")))
    fig.update_xaxes(title_text="x (mm, workpiece)")
    fig.update_yaxes(scaleanchor="x", scaleratio=1, title_text="y (mm, workpiece)")
    show(fig, f"{r.name} — the path, coloured by {cbar}",
         f'{int(eng.sum())} of {len(a)} nodes engaged, one every '
         f'{r.summary["ds_mm"]:g} mm · hollow = out of the cut', height=380)


fig_map(r, by="growth")

## 3 · Stability along the path

One row per quantity — growth rate, the critical depth, the frequency the mode
sits at. The wash marks where the tool is engaged; the dashed lines are the two
thresholds that matter: zero growth, and the depth actually being run.

In [ ]:
def fig_stability(r):
    a = r.along
    s = a["s_mm"].to_numpy(float)
    eng = a["engaged"].to_numpy(bool)
    g = a["growth_rate_1_s"].to_numpy(float)
    unst = eng & (g > 0)

    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.075,
                        subplot_titles=("growth rate (1/s) — below zero is stable",
                                        "ap_crit (mm) — the depth this node survives",
                                        "chatter mode (Hz)"))
    fig.add_trace(go.Scatter(x=s, y=g, mode="lines", name="growth",
                             line=dict(color=P["series"][0], width=W),
                             hovertemplate="%{y:+.2f} 1/s<extra>growth</extra>"), row=1, col=1)
    if unst.any():
        fig.add_trace(go.Scatter(x=s[unst], y=g[unst], mode="markers", name="unstable",
                                 marker=dict(size=9, color=P["critical"],
                                             line=dict(color=P["surface"], width=1)),
                                 hovertemplate="%{y:+.2f} 1/s<extra>unstable</extra>"),
                      row=1, col=1)
    fig.add_hline(y=0, line=dict(color=P["axis"], width=1, dash="dash"), row=1, col=1)

    fig.add_trace(go.Scatter(x=s, y=a["ap_crit_mm"], mode="lines", name="ap_crit",
                             line=dict(color=P["series"][1], width=W),
                             hovertemplate="%{y:.2f} mm<extra>ap_crit</extra>"), row=2, col=1)
    fig.add_hline(y=r.ap_mm, line=dict(color=P["axis"], width=1, dash="dash"),
                  annotation_text=f"running at ap = {r.ap_mm:g} mm",
                  annotation_position="bottom right",
                  annotation_font=dict(size=11, color=P["muted"]), row=2, col=1)
    # ap_crit runs away where the tool is barely in the cut, so it needs a log axis
    # to show both that and the plateau the run is actually judged on. dtick=1 keeps
    # the ticks on the decades — plotly's default log minor labels read as noise.
    fig.update_yaxes(type="log", dtick=1, row=2, col=1)

    fig.add_trace(go.Scatter(x=s, y=a["mode_hz"], mode="lines", name="mode",
                             line=dict(color=P["series"][2], width=W),
                             hovertemplate="%{y:.1f} Hz<extra>mode</extra>"), row=3, col=1)
    for hz in np.asarray(r.plant["modes_hz"], float):
        fig.add_hline(y=hz, line=dict(color=P["grid"], width=1), row=3, col=1)

    subtitles(fig, 3)
    for row in (1, 2, 3):
        shade(fig, s, eng, label="engaged" if row == 1 else None, row=row, col=1)
    fig.update_xaxes(title_text="s along the path (mm)", row=3, col=1)
    show(fig, f"{r.name} — linear stability, node by node",
         f'trimmed verdict: {"unstable" if r.summary["pred_unstable_trim"] else "stable"}'
         f' · {r.summary["pred_unstable_fraction"] * 100:.0f}% of engaged nodes above zero'
         f' · grey lines in the bottom row are the plant modes',
         height=700, showlegend=False, hovermode="x unified")


fig_stability(r)

## 4 · The linear force along the path

`F0(s)` is the surrogate's steady cutting force at each node — what the
feedforward cancels, and what section 8 checks the cutting engine against.

In [ ]:
def fig_force_path(r):
    a = r.along
    s = a["s_mm"].to_numpy(float)
    fig = go.Figure()
    for i, (c, lab) in enumerate([("F0_x_N", "F0 x"), ("F0_y_N", "F0 y"),
                                  ("F0_z_N", "F0 z"), ("F0_mag_N", "|F0|")]):
        fig.add_trace(go.Scatter(x=s, y=a[c], mode="lines", name=lab,
                                 line=dict(color=P["series"][i], width=W,
                                           dash="dot" if c == "F0_mag_N" else "solid"),
                                 hovertemplate="%{y:.1f} N<extra>" + lab + "</extra>"))
    fig.add_hline(y=0, line=dict(color=P["axis"], width=1))
    shade(fig, s, a["engaged"].to_numpy(bool), label="engaged")
    fig.update_xaxes(title_text="s along the path (mm)")
    fig.update_yaxes(title_text="force (N, workpiece frame)")
    show(fig, f"{r.name} — the linear force model along the path",
         "F0(s) from the surrogate at each node", height=430, hovermode="x unified")


fig_force_path(r)

## 5 · The simulated pass

What actually happened: how far the tool tip sat from where it was commanded,
and the force that put it there. Both rows share the time axis; the wash is the
stretch where the engine reports force. Pass `t0`/`t1` to zoom into a window.

In [ ]:
def fig_pass(r, t0=None, t1=None):
    d = r.ts
    if t0 is not None or t1 is not None:
        d = d[(d["t_s"] >= (-np.inf if t0 is None else t0))
              & (d["t_s"] <= (np.inf if t1 is None else t1))]
    t = d["t_s"].to_numpy(float)
    # Tooth passing is well above what the decimated CSV can resolve, so the raw
    # force trace is aliased hash. Averaging it over one revolution is honest and
    # readable; section 8 does the same thing properly, revolution by revolution.
    rev_hz = r.summary["spindle_rpm"] / 60.0 * r.config["mill"]["n_teeth"]
    win = r.rev_window
    cut = d["F_mag_N"].rolling(win, center=True, min_periods=1).max().to_numpy() > 0

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.09,
                        subplot_titles=("deviation from the commanded path (µm)",
                                        "cutting force at the tool tip (N, workpiece frame) "
                                        f"— averaged over one revolution ({win} samples)"))
    for i, (c, lab) in enumerate([("dev_x_um", "dev x"), ("dev_y_um", "dev y"),
                                  ("dev_mag_um", "|dev|")]):
        fig.add_trace(go.Scatter(x=t, y=d[c], mode="lines", name=lab, legendgroup="dev",
                                 line=dict(color=P["series"][i], width=W,
                                           dash="dot" if c == "dev_mag_um" else "solid"),
                                 hovertemplate="%{y:.1f} µm<extra>" + lab + "</extra>"),
                      row=1, col=1)
    # |F| is the norm of the averaged components, not the average of the norms —
    # otherwise it would sit well above the plateau section 8 reports.
    comp = {c: d[c].rolling(win, center=True, min_periods=1).mean()
            for c in ("Fx_N", "Fy_N", "Fz_N")}
    mean = dict(comp)
    mean["F_mag_N"] = np.sqrt(sum(v ** 2 for v in comp.values()))
    for i, (c, lab) in enumerate([("Fx_N", "Fx"), ("Fy_N", "Fy"), ("Fz_N", "Fz"),
                                  ("F_mag_N", "|F|")]):
        y = mean[c]
        fig.add_trace(go.Scatter(x=t, y=y, mode="lines", name=lab, legend="legend2",
                                 line=dict(color=P["series"][i], width=W,
                                           dash="dot" if c == "F_mag_N" else "solid"),
                                 hovertemplate="%{y:.0f} N<extra>" + lab + "</extra>"),
                      row=2, col=1)
    second_legend(fig, y=0.45)
    fig.add_hline(y=r.summary["sim_dc_dev_um"], line=dict(color=P["axis"], width=1, dash="dash"),
                  annotation_text=f'DC {r.summary["sim_dc_dev_um"]:.1f} µm',
                  annotation_position="top left",
                  annotation_font=dict(size=11, color=P["muted"]), row=1, col=1)
    subtitles(fig, 2)
    for row in (1, 2):
        shade(fig, t, cut, label="cutting" if row == 1 else None, row=row, col=1)
    fig.update_xaxes(title_text="t (s)", row=2, col=1)
    show(fig, f"{r.name} — the pass as simulated",
         f'{r.summary["sim_engaged_s"]:.2f} s engaged · peak deviation '
         f'{r.summary["sim_peak_dev_um"]:.1f} µm · peak |F| '
         f'{r.summary["sim_peak_force_N"]:.0f} N · CSV decimated '
         f'{r.summary["csv_decimate"]}x to {1.0 / r.dt:.0f} Hz, below the '
         f'{rev_hz:.0f} Hz tooth passing', height=660, hovermode="x unified")


fig_pass(r)

## 6 · Where the tool actually went

The commanded path and the tool tip, with the deviation multiplied so it is
visible at all — at 1:1 it is a few tens of microns on a 100 mm cut.

In [ ]:
def fig_track(r, gain=200):
    d = r.ts
    cx, cy = d["cmd_x_mm"].to_numpy(float), d["cmd_y_mm"].to_numpy(float)
    ex = cx + gain * (d["tcp_x_mm"].to_numpy(float) - cx)
    ey = cy + gain * (d["tcp_y_mm"].to_numpy(float) - cy)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=cx, y=cy, mode="lines", name="commanded",
                             line=dict(color=P["muted"], width=1, dash="dash"), hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=ex, y=ey, mode="lines",
                             name=f"tool tip, deviation ×{gain:g}",
                             line=dict(color=P["series"][0], width=W),
                             customdata=np.c_[d["t_s"], d["dev_mag_um"], d["F_mag_N"]],
                             hovertemplate=("t %{customdata[0]:.3f} s<br>"
                                            "|dev| %{customdata[1]:.1f} µm · "
                                            "|F| %{customdata[2]:.0f} N<extra></extra>")))
    fig.update_xaxes(title_text="x (mm, workpiece)")
    fig.update_yaxes(scaleanchor="x", scaleratio=1, title_text="y (mm, workpiece)")
    show(fig, f"{r.name} — the tool tip against the command",
         f'deviation exaggerated {gain:g}× · true |dev| peaks at '
         f'{r.summary["sim_peak_dev_um"]:.1f} µm', height=450)


fig_track(r, gain=200)

## 7 · What the deviation is made of

Amplitude spectrum of the deviation over the engaged stretch, mean removed and
Hann-windowed. The fitted chatter frequency and the plant's own modes are
marked — a peak that lines up with a mode is the arm ringing, not the cut.

In [ ]:
def fig_spectrum(r, f_max=60.0):
    d = r.ts[r.cutting]
    if len(d) < 16:
        print("not enough engaged samples for a spectrum")
        return
    fig = go.Figure()
    for i, (c, lab) in enumerate([("dev_x_um", "dev x"), ("dev_y_um", "dev y")]):
        y = d[c].to_numpy(float)
        y = y - y.mean()
        n = len(y)
        amp = 2.0 * np.abs(np.fft.rfft(y * np.hanning(n))) / (n * 0.5)
        f = np.fft.rfftfreq(n, d=r.dt)
        m = f <= f_max
        fig.add_trace(go.Scatter(x=f[m], y=amp[m], mode="lines", name=lab,
                                 line=dict(color=P["series"][i], width=W),
                                 hovertemplate="%{y:.2f} µm<extra>" + lab + "</extra>"))
    for hz in np.asarray(r.plant["modes_hz"], float):
        fig.add_vline(x=hz, line=dict(color=P["grid"], width=1))
    # The two marks sit within a couple of Hz of each other, so they are staggered
    # vertically rather than left/right.
    fig.add_vline(x=r.summary["sim_chatter_hz"], line=dict(color=P["muted"], width=1, dash="dash"),
                  annotation_text=f'fitted {r.summary["sim_chatter_hz"]:.1f} Hz',
                  annotation_position="top right", annotation_yshift=-2,
                  annotation_font=dict(size=11, color=P["muted"]))
    fig.add_vline(x=r.summary["pred_mode_hz"], line=dict(color=P["muted"], width=1, dash="dot"),
                  annotation_text=f'predicted {r.summary["pred_mode_hz"]:.1f} Hz',
                  annotation_position="top left", annotation_yshift=-22,
                  annotation_font=dict(size=11, color=P["muted"]))
    fig.update_xaxes(title_text="frequency (Hz)")
    fig.update_yaxes(title_text="amplitude (µm)")
    show(fig, f"{r.name} — deviation spectrum over the cut",
         f'{len(d)} engaged samples at {1.0 / r.dt:.0f} Hz · grey lines are the plant modes '
         f'({", ".join(f"{h:.1f}" for h in np.asarray(r.plant["modes_hz"], float))} Hz)',
         height=430, hovermode="x unified")


fig_spectrum(r)

## 8 · Does the engine agree with the linear model?

One point per spindle revolution: the revolution-averaged force out of the
cutting engine against `F0` from the surrogate. The run is scored on the plateau
and on the entry/exit revolutions separately, since only the ends see a changing
engagement.

In [ ]:
def fig_revolutions(r):
    d = r.revs[r.revs["cutting"].astype(bool)]
    s = d["s_mm"].to_numpy(float)
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.09,
                        subplot_titles=("revolution-averaged |F| (N)",
                                        "error (N) — how big, and how much of it is magnitude"))
    fig.add_trace(go.Scatter(x=s, y=d["F_sim_mag_N"], mode="lines", name="cutting engine",
                             line=dict(color=P["series"][0], width=W),
                             hovertemplate="%{y:.1f} N<extra>cutting engine</extra>"), row=1, col=1)
    fig.add_trace(go.Scatter(x=s, y=d["F_lin_mag_N"], mode="lines", name="linear F0",
                             line=dict(color=P["series"][1], width=W),
                             hovertemplate="%{y:.1f} N<extra>linear F0</extra>"), row=1, col=1)
    # |error vector| is always positive; the signed magnitude difference says which
    # way the engine ran, and the gap between the two is direction error.
    fig.add_trace(go.Scatter(x=s, y=d["err_mag_N"], mode="lines", name="|error vector|",
                             line=dict(color=P["series"][2], width=W), legend="legend2",
                             hovertemplate="%{y:.2f} N<extra>|error vector|</extra>"), row=2, col=1)
    fig.add_trace(go.Scatter(x=s, y=d["F_sim_mag_N"] - d["F_lin_mag_N"], mode="lines",
                             name="|F| difference", legend="legend2",
                             line=dict(color=P["series"][3], width=W, dash="dot"),
                             hovertemplate="%{y:+.2f} N<extra>|F| difference</extra>"), row=2, col=1)
    fig.add_hline(y=0, line=dict(color=P["axis"], width=1, dash="dash"), row=2, col=1)
    second_legend(fig, y=0.45)
    subtitles(fig, 2)
    fig.update_xaxes(title_text="s along the path (mm)", row=2, col=1)
    s_ = r.summary
    show(fig, f"{r.name} — cutting engine vs the linear surrogate",
         f'plateau {s_["plateau_mag_err"] * 100:+.2f} % magnitude, '
         f'{s_["plateau_angle_err_deg"]:.2f}° direction ({s_["plateau_n_rev"]} revs)'
         f' · entry+exit {s_["ends_mag_err"] * 100:+.2f} %, '
         f'{s_["ends_angle_err_deg"]:.2f}° ({s_["ends_n_rev"]} revs)',
         height=580, hovermode="x unified")


fig_revolutions(r)

## 9 · The feedforward

What the compensation was asked to carry, and how far `G(0)·F0` says the tool
would have been pushed had nobody carried it. Prints a note instead on a
`_plain` run.

In [ ]:
def fig_feedforward(r):
    if r.ff is None:
        print(f"{r.name} ran without feedforward — nothing to show")
        return
    ff = r.ff
    s = np.asarray(ff["s_mm"], float)
    eng = np.asarray(ff["engaged"], bool)
    F0 = np.linalg.norm(np.asarray(ff["F0_w"], float), axis=1)
    carried = np.linalg.norm(np.asarray(ff["carried_w"], float), axis=1)
    offset_um = np.linalg.norm(np.asarray(ff["offset_w"], float), axis=1) * 1e6

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.09,
                        subplot_titles=("force (N)",
                                        "G(0)·F0 — the offset it prevents (µm)"))
    fig.add_trace(go.Scatter(x=s, y=F0, mode="lines", name="|F0| asked for",
                             line=dict(color=P["series"][0], width=W),
                             hovertemplate="%{y:.1f} N<extra>|F0| asked for</extra>"), row=1, col=1)
    fig.add_trace(go.Scatter(x=s, y=carried, mode="lines", name="|carried| on the motors",
                             line=dict(color=P["series"][1], width=W, dash="dash"),
                             hovertemplate="%{y:.1f} N<extra>carried</extra>"), row=1, col=1)
    fig.add_trace(go.Scatter(x=s, y=offset_um, mode="lines", name="offset", showlegend=False,
                             line=dict(color=P["series"][2], width=W),
                             hovertemplate="%{y:.1f} µm<extra>offset</extra>"), row=2, col=1)
    subtitles(fig, 2)
    shade(fig, s, eng, label="engaged", row=1, col=1)
    shade(fig, s, eng, row=2, col=1)
    fig.update_xaxes(title_text="s along the path (mm)", row=2, col=1)
    s_ = r.summary
    show(fig, f"{r.name} — the DC compensation",
         f'axes {str(ff["axes"])}, gain {float(ff["gain"]):g} · would have held the tool up to '
         f'{s_["ff_offset_max_um"]:.0f} µm off nominal · residual DC after it: '
         f'{s_["sim_dc_dev_um"]:.1f} µm', height=580, hovermode="x unified")


fig_feedforward(r)

## 10 · The plant

The receptance of the linearised arm at the tool tip, straight out of
`plant.npz`: `G(jw) = C (jwI − A)⁻¹ B + D`, converted to microns per newton. The
diagonal terms are what a force along one axis does to that same axis.

In [ ]:
def frf(plant, f_hz):
    """G(jw) for each frequency in `f_hz`, shape (len(f), 3, 3), SI (m/N)."""
    A, B, C, D = (np.asarray(plant[k], float) for k in "ABCD")
    I = np.eye(A.shape[0])
    return np.stack([C @ np.linalg.solve(1j * 2 * np.pi * f * I - A, B) + D for f in f_hz])


def fig_plant(r, f_lo=0.5, f_hi=100.0, n=1200):
    f = np.geomspace(f_lo, f_hi, n)
    G = frf(r.plant, f) * 1e6                       # m/N -> um/N
    fig = go.Figure()
    for i, ax in enumerate("xyz"):
        fig.add_trace(go.Scatter(x=f, y=np.abs(G[:, i, i]), mode="lines", name=f"G{ax}{ax}",
                                 line=dict(color=P["series"][i], width=W),
                                 hovertemplate="%{y:.3f} µm/N<extra>" + f"G{ax}{ax}" + "</extra>"))
    for k, (hz, z) in enumerate(zip(np.asarray(r.plant["modes_hz"], float),
                                    np.asarray(r.plant["damping"], float))):
        # Shapes on a log axis take log10 coordinates, not Hz — and the modes sit
        # close together, so the labels are staggered downwards.
        fig.add_vline(x=np.log10(hz), line=dict(color=P["grid"], width=1),
                      annotation_text=f"{hz:.1f} Hz · ζ {z:.3f}",
                      annotation_position="top right", annotation_yshift=-2 - 20 * k,
                      annotation_font=dict(size=11, color=P["muted"]))
    ticks = [t for t in (0.1, 0.2, 0.5, 1, 2, 5, 10, 20, 50, 100, 200, 500, 1000)
             if f_lo <= t <= f_hi]
    fig.update_xaxes(type="log", title_text="frequency (Hz)",
                     range=[np.log10(f_lo), np.log10(f_hi)],
                     tickmode="array", tickvals=ticks, ticktext=[f"{t:g}" for t in ticks])
    fig.update_yaxes(type="log", dtick=1, title_text="compliance (µm/N)")
    G0 = np.abs(frf(r.plant, [1e-9])[0]) * 1e6
    show(fig, f"{r.name} — tool-tip receptance, {str(r.plant['frame'])} frame",
         f'{str(r.plant["name"])} · static compliance '
         f'{G0[0, 0]:.2f} / {G0[1, 1]:.2f} / {G0[2, 2]:.2f} µm/N on x/y/z',
         height=450, hovermode="x unified")


fig_plant(r)

## 11 · Compare runs

Everything above looks at one folder; this puts them side by side — a `_comp`
run against its `_plain` twin, say. The table is the full scalar record; the
charts are the three comparisons worth looking at.

In [ ]:
KEYS = ["compensated", "ff_applied", "axial_depth_mm", "spindle_rpm", "fz_mm",
        "pred_growth_max_trim_1_s", "pred_ap_crit_trim_mm", "pred_mode_hz",
        "sim_growth_1_s", "sim_chatter_hz", "sim_dc_dev_um", "sim_ac_rms_um",
        "sim_peak_dev_um", "sim_peak_force_N", "plateau_mag_err", "ends_mag_err",
        "agree_trim", "growth_err_trim_1_s"]

runs = {n: load_run(n) for n in available()}
table = pd.DataFrame({n: {k: x.summary.get(k) for k in KEYS} for n, x in runs.items()})
display(table.style.format(precision=3).set_caption("summary.json, side by side"))

In [ ]:
def fig_compare_dev(runs):
    fig = go.Figure()
    for i, (n, x) in enumerate(runs.items()):
        fig.add_trace(go.Scatter(x=x.ts["t_s"], y=x.ts["dev_mag_um"], mode="lines", name=n,
                                 line=dict(color=P["series"][i % len(P["series"])], width=W,
                                           dash=DASHES[i % len(DASHES)]),
                                 hovertemplate="%{y:.1f} µm<extra>" + n + "</extra>"))
    fig.update_xaxes(title_text="t (s)")
    fig.update_yaxes(title_text="|deviation| (µm)")
    show(fig, "Deviation from the commanded path, run by run",
         "the same cut, different compensation", height=430, hovermode="x unified")


def fig_compare_bars(runs, keys=(("sim_dc_dev_um", "DC"), ("sim_ac_rms_um", "AC rms"),
                                 ("sim_peak_dev_um", "peak"))):
    fig = go.Figure()
    for i, (n, x) in enumerate(runs.items()):
        fig.add_trace(go.Bar(x=[lab for _, lab in keys], y=[x.summary[k] for k, _ in keys], name=n,
                             marker=dict(color=P["series"][i % len(P["series"])],
                                         line=dict(color=P["surface"], width=2)),
                             text=[f'{x.summary[k]:.1f}' for k, _ in keys],
                             textposition="outside", textfont=dict(color=P["ink2"], size=11),
                             hovertemplate="%{y:.1f} µm<extra>" + n + "</extra>"))
    top = max(x.summary[k] for x in runs.values() for k, _ in keys)
    fig.update_yaxes(title_text="deviation (µm)", range=[0, top * 1.15])  # room for the labels
    show(fig, "Deviation, decomposed",
         "all three in microns, so one axis carries them", height=410,
         barmode="group", bargap=0.35, bargroupgap=0.06)


def fig_compare_growth(runs):
    fig = go.Figure()
    for i, (n, x) in enumerate(runs.items()):
        fig.add_trace(go.Scatter(x=x.along["s_mm"], y=x.along["growth_rate_1_s"], mode="lines",
                                 name=n, hovertemplate="%{y:+.2f} 1/s<extra>" + n + "</extra>",
                                 line=dict(color=P["series"][i % len(P["series"])], width=W,
                                           dash=DASHES[i % len(DASHES)])))
    fig.add_hline(y=0, line=dict(color=P["axis"], width=1, dash="dash"),
                  annotation_text="unstable above", annotation_position="top right",
                  annotation_font=dict(size=11, color=P["muted"]))
    fig.update_xaxes(title_text="s along the path (mm)")
    fig.update_yaxes(title_text="growth rate (1/s)")
    show(fig, "Predicted growth rate along the path, run by run",
         "below the grey line the cut is stable · runs that differ only in their "
         "compensation predict the same thing, so their curves coincide — the dashes "
         "tell them apart", height=430, hovermode="x unified")


fig_compare_dev(runs)
fig_compare_bars(runs)
fig_compare_growth(runs)

## Scratch

`r.coupled` is the full simulation — every sample, every joint — for anything
the sections above do not cover.

| key | shape | what it is |
|---|---|---|
| `t`, `q_hist`, `theta_cmd` | (N,), (N,6), (N,6) | time, joint state, commanded joints |
| `force_i`, `force_w` | (N,3) | cutting force, inertial and workpiece frames |
| `tcp_error_w_um`, `deflection_w_um` | (N,3) | tip error and deflection, microns |
| `fk_hist`, `nominal_i`, `T_iw` | (N,6), (N,3), (4,4) | forward kinematics, nominal path, frame |

In [ ]:
for k, v in r.coupled.items():
    print(f"{k:20s} {str(v.dtype):9s} {v.shape}")